# Sketch 2.4 - standalone, portable notebook

Self-contained extract of **sketch_2_4_map** from the combined *9 sketches*
notebook. The sketch code is unchanged; the shared setup it needs
(imports, data loading, department outlines) is included below. Chart data is **inlined** (no external
`altair-data` files) so it renders on Linux, macOS and Windows alike.

The searchable name universe is restricted to the top-40 names by total (plus the
sketch's defaults) so the inlined data stays small; the default view is
unchanged.

## Shared setup *(unchanged)*

In [ ]:
import altair as alt
import pandas as pd
import numpy as np
import geopandas as gpd  # conda install -c conda-forge geopandas
import json

# Inline the chart data so every HTML output is SELF-CONTAINED and renders on
# all platforms (incl. macOS): external altair-data-*.json URL files are not
# fetched by many notebook frontends, which left the charts blank on Mac. The
# search sketches (1.13 / 2.4 / 2.6) restrict their name universe below so the
# inlined tables stay small.
alt.data_transformers.enable('default')
alt.data_transformers.disable_max_rows()
pass

In [ ]:
# Memory-lean load: another heavy job is running on this machine (~2.5 GB free),
# so we read the CSV with categorical dtypes (15k name strings stored once), derive
# the numeric year from the 122 'annais' categories instead of 3.5M strings, then
# relax categories back to plain objects (the strings stay shared) so every later
# groupby keeps its normal semantics.
raw = pd.read_csv('dpt2020.csv', sep=';',
                  dtype={'sexe': 'int8', 'preusuel': 'category',
                         'annais': 'category', 'dpt': 'category', 'nombre': 'int32'})
cat_years = pd.to_numeric(pd.Index(raw['annais'].cat.categories), errors='coerce').astype('float32')
raw['year'] = np.asarray(cat_years)[raw['annais'].cat.codes]
raw.drop(columns=['annais'], inplace=True)
raw['decade'] = (raw['year'] // 10 * 10)
raw['preusuel'] = raw['preusuel'].astype(object)  # shared strings, plain groupbys
raw['dpt'] = raw['dpt'].astype(object)

records = raw[raw.dpt != 'XX'].copy()
named = records[records.preusuel != '_PRENOMS_RARES'].copy()

dept_total = records.groupby('dpt', as_index=False)['nombre'].sum().rename(
    columns={'nombre': 'dept_total'})

ALL_NAMES = sorted(named.preusuel.unique().tolist())  # full dropdown option list
DECADES = list(range(1900, 2021, 10))
# Shared period options for charts that need an all-years aggregate (2.4).
DECADE_OPTS = [0] + DECADES
DECADE_LABELS = ['All years'] + [str(d) for d in DECADES]
SEX_OPTIONS = ['Mixed', 'Boys', 'Girls']
SEX_LABELS = ['mixed', 'boys', 'girls']

import gc
del raw
gc.collect()

print(f'{len(named):,} named rows; {len(ALL_NAMES):,} distinct names.')
named.sample(3)


## Name-universe restriction *(portability/memory)*

In [ ]:
# Portability + memory: keep the INLINED data small by limiting the name
# universe to names that are in the top 40 by total births, always unioned with this sketch's own
# default names so the default view is unchanged.
import gc
_EXTRA = [x for x in ['JEAN'] if x in set(named['preusuel'])]
_universe = set(named.groupby('preusuel')['nombre'].sum().nlargest(40).index)
POPULAR_NAMES = sorted(_universe.union(_EXTRA))
named = named[named['preusuel'].isin(POPULAR_NAMES)].copy()
ALL_NAMES = POPULAR_NAMES
gc.collect()
print(f'Universe: {len(POPULAR_NAMES):,} names (are in the top 40 by total births + defaults); {len(named):,} rows kept.')

In [ ]:
depts = gpd.read_file('departements-version-simplifiee.geojson')
_corsica = depts['code'].isin(['2A', '2B'])
depts = pd.concat([
    depts[~_corsica],
    gpd.GeoDataFrame({'code': ['20'], 'nom': ['Corse']},
                     geometry=[depts.loc[_corsica, 'geometry'].union_all()],
                     crs=depts.crs),
], ignore_index=True)
METRO = set(depts['code'])

dept_name = named.groupby(['dpt', 'preusuel'], as_index=False)['nombre'].sum()
dept_name = dept_name.merge(dept_total, on='dpt')
dept_name['per_1000'] = (1000 * dept_name['nombre'] / dept_name['dept_total']).round(3)
dept_name = dept_name.merge(depts[['code', 'nom']], left_on='dpt', right_on='code',
                            how='left').drop(columns=['code', 'dept_total'])

geo_features = alt.InlineData(values=json.loads(depts.to_json()),
                              format=alt.DataFormat(property='features'))
FRANCE_PROJECTION = dict(type='conicConformal', rotate=[-3, 0],
                         center=[0, 46.5], parallels=[44, 49])
print(f'{len(dept_name):,} (department, name) pairs embedded.')
dept_name.sample(3)

## The sketch

### Sketch 2.4 - Map of France: dropdown over the full 15k list (+ search, period gauge)

Pick any of the ~15,270 names from the dropdown (type-ahead works in the
browser); typing in the search box overrides the dropdown. The period is a
sliding gauge: drag the slider through the century (decade steps - the honest
granularity of the underlying ~980k-row department x name x decade table, served as
an external JSON referenced by URL); tick All years to aggregate 1900-2020.
By default, All years is unchecked and the chart opens on the selected decade.
The active name is printed on the map; hovering outlines a department in orange.


In [ ]:
# (department, name, decade) rate table -> external JSON referenced by URL.
# ~745k rows: kept OUT of the spec (the json transformer would re-serialise it per
# layer); pandas writes it once and both layers point at the same file.
_slim4 = records.loc[records['decade'].notna(), ['dpt', 'decade', 'nombre']]
dec_tot4 = (_slim4.groupby(['dpt', 'decade'], as_index=False)['nombre'].sum()
            .rename(columns={'nombre': 'tot'}))
dn_dec = (named.dropna(subset=['decade'])
          .groupby(['dpt', 'preusuel', 'decade'], as_index=False)['nombre'].sum()
          .merge(dec_tot4, on=['dpt', 'decade']))
dn_dec['per_1000'] = (1000 * dn_dec['nombre'] / dn_dec['tot']).round(2)
dn_dec = dn_dec.drop(columns=['tot'])
dn_all = dept_name.copy()
dn_all['decade'] = 0
dn_dec = pd.concat([dn_dec[['dpt', 'preusuel', 'decade', 'nombre', 'per_1000']],
                    dn_all[['dpt', 'preusuel', 'decade', 'nombre', 'per_1000']]],
                   ignore_index=True)
dn_dec['decade'] = dn_dec['decade'].astype(int)
dn_dec = dn_dec.merge(depts[['code', 'nom']], left_on='dpt', right_on='code',
                      how='left').drop(columns='code')
print(f'{len(dn_dec):,} (dept, name, period) rows kept inline (no external file).')
del _slim4, dn_all

name_dd = alt.param(name='name_dd', value='JEAN',
                    bind=alt.binding_select(options=ALL_NAMES, name='Name (dropdown)  '))
name_q2 = alt.param(name='name_q2', value='',
                    bind=alt.binding(input='search', name='Search (overrides)  ',
                                     placeholder='e.g. LUCIEN'))
all_yrs = alt.param(name='all_yrs', value=False,
                    bind=alt.binding_checkbox(name='All years (1900-2020)  '))
decade_g = alt.param(name='decade_g', value=1980,
                     bind=alt.binding_range(min=1900, max=2020, step=10,
                                            name='Decade slider  '))
picked = ("(name_q2 != '' ? upper(datum.preusuel) == upper(name_q2)"
          " : datum.preusuel == name_dd)"
          " && (all_yrs ? datum.decade == 0 : datum.decade == decade_g)")
dept_hover = alt.selection_point(fields=['nom'], on='pointerover', clear='pointerout', empty=False)

base_map = alt.Chart(geo_features).mark_geoshape(
    fill='#efefef', stroke='white', strokeWidth=0.4).project(**FRANCE_PROJECTION)
choro = alt.Chart(dn_dec).transform_filter(picked).transform_lookup(
    lookup='dpt', from_=alt.LookupData(geo_features, key='properties.code', fields=['type', 'geometry']),
).mark_geoshape().encode(
    color=alt.Color('per_1000:Q', scale=alt.Scale(scheme='viridis'),
                    legend=alt.Legend(title=['Births', 'per 1000'])),
    stroke=alt.condition(dept_hover, alt.value('#ff7f0e'), alt.value('white')),
    strokeWidth=alt.condition(dept_hover, alt.value(2.5), alt.value(0.4)),
    tooltip=[alt.Tooltip('nom:N', title='Department'), alt.Tooltip('preusuel:N', title='Name'),
             alt.Tooltip('nombre:Q', title='Births', format=','),
             alt.Tooltip('per_1000:Q', title='Births / 1000', format='.2f')],
).project(**FRANCE_PROJECTION)
name_tag = (alt.Chart(dn_dec).transform_filter(picked)
            .transform_aggregate(n='count()', groupby=['preusuel'])
            .transform_calculate(lon='-4.6', lat='51.3')
            .mark_text(align='left', fontSize=17, fontWeight='bold', color='#222')
            .encode(longitude='lon:Q', latitude='lat:Q', text='preusuel:N').project(**FRANCE_PROJECTION))
sketch_2_4 = (base_map + choro + name_tag).add_params(name_dd, name_q2, all_yrs, decade_g, dept_hover).properties(
    width=560, height=520,
    title='2.4 - Where is a name most common? (any name; slide the decade)')

# PNG snapshot: vl-convert cannot fetch URL data (it inlines DataFrames instead),
# so the static export is rendered from the default state (JEAN, 1980) as a
# small in-memory slice; the displayed chart above stays URL-backed + interactive.
_png_slice = dn_dec[(dn_dec.preusuel == 'JEAN') & (dn_dec.decade == 1980)]
_png_choro = alt.Chart(_png_slice).transform_lookup(
    lookup='dpt', from_=alt.LookupData(geo_features, key='properties.code', fields=['type', 'geometry']),
).mark_geoshape(stroke='white', strokeWidth=0.4).encode(
    color=alt.Color('per_1000:Q', scale=alt.Scale(scheme='viridis'),
                    legend=alt.Legend(title=['Births', 'per 1000'])),
).project(**FRANCE_PROJECTION)
_png_tag = alt.Chart(pd.DataFrame({'lon': [-4.6], 'lat': [51.3], 't': ['JEAN']})).mark_text(
    align='left', fontSize=17, fontWeight='bold', color='#222').encode(
    longitude='lon:Q', latitude='lat:Q', text='t:N').project(**FRANCE_PROJECTION)
(base_map + _png_choro + _png_tag).properties(
    width=560, height=520,
    title='2.4 - Where is a name most common? (any name; slide the decade)'
).save('sketch_2_4_map_select.png', ppi=120)
sketch_2_4
